# Garmin Connect — Download dati

Questo notebook scarica in modo sequenziale e trasparente tutti i dati rilevanti da **Garmin Connect** e li salva in un singolo file JSON strutturato.

### Dati scaricati (ultimi 30 giorni)
| Sezione | Contenuto |
|---|---|
| `activities` | Lista attività con metriche principali |
| `splits` | Lap per ogni corsa |
| `resting_hr` | FC a riposo giornaliera |
| `body_battery` | Carica/scarica body battery |
| `sleep` | Dati sonno dettagliati |
| `hrv` | Variabilità della frequenza cardiaca |
| `stress` | Livello di stress medio e picco |
| `steps` | Passi giornalieri |
| `training_plan` | Workout pianificati (−7 / +30 giorni) |
| `personal_records` | Record personali snapshot attuale |
| `fitness_age` | Età fitness e metriche correlate |

## 1 — Percorsi e configurazione

Definiamo i percorsi fissi e carichiamo le variabili d'ambiente:
- **`TOKEN_DIR`** — directory dove `garth` salva/legge i token OAuth di Garmin
- **`ENV_PATH`** — file `.env` con le credenziali Garmin (`GARMIN_EMAIL`, `GARMIN_PASSWORD`)
- **`OUTPUT_PATH`** — file JSON di output, nominato con la data del download (`garmin_YYYY-MM-DD.json`)
- **`DAYS`** — finestra temporale (giorni nel passato) per i dati storici

In [4]:
from pathlib import Path
from datetime import date, timedelta
from dotenv import load_dotenv

DAYS  = 30
TODAY = date.today()

REPO_ROOT   = Path.cwd().parent
TOKEN_DIR   = REPO_ROOT / ".garmin_token"
ENV_PATH    = REPO_ROOT / ".env"
OUTPUT_PATH = REPO_ROOT / "data" / f"garmin_{TODAY.isoformat()}.json"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
loaded = load_dotenv(ENV_PATH)

print(f"Repo root  : {REPO_ROOT}")
print(f"Token dir  : {TOKEN_DIR}")
print(f"Env file   : {ENV_PATH}  {'[OK]' if loaded else '[NON TROVATO]'}")
print(f"Output file: {OUTPUT_PATH}")
print(f"Finestra   : ultimi {DAYS} giorni ({(TODAY - timedelta(days=DAYS)).isoformat()} → {TODAY.isoformat()})")

Repo root  : c:\Users\feder\Script
Token dir  : c:\Users\feder\Script\.garmin_token
Env file   : c:\Users\feder\Script\.env  [OK]
Output file: c:\Users\feder\Script\data\garmin_2026-05-10.json
Finestra   : ultimi 30 giorni (2026-04-10 → 2026-05-10)


## 2 — Autenticazione

Il login avviene tramite **token cached**: se il token è già presente nella `TOKEN_DIR`, viene usato direttamente senza richiedere le credenziali.

Al primo avvio (o se il token è scaduto), le credenziali vengono lette dalle variabili d'ambiente caricate nella cella precedente. Se il tuo account ha **MFA attivo**, il codice OTP verrà chiesto a schermo.

In [7]:
import os
import garth
from getpass import getpass
from garminconnect import Garmin

email    = os.getenv("GARMIN_EMAIL")    or input("Garmin email: ")
password = os.getenv("GARMIN_PASSWORD") or getpass("Garmin password: ")

client = Garmin(email, password, prompt_mfa=lambda: input("Codice MFA: "))

try:
    client.login(str(TOKEN_DIR))
    print(f"Login OK — token cached in {TOKEN_DIR}")
except Exception as e:
    print(f"Token non valido o assente ({type(e).__name__}) — login fresco...")
    client.login()
    garth.save(str(TOKEN_DIR))
    print(f"Login OK — token salvato in {TOKEN_DIR}")

Login OK — token cached in c:\Users\feder\Script\.garmin_token


## 3 — Inizializzazione snapshot e funzioni di supporto

Creiamo la struttura base del dizionario che raccoglierà tutti i dati scaricati e definiamo:
- **`save()`** — scrive lo snapshot su disco ogni volta che una sezione viene completata
- **`_pace()`** — converte velocità m/s in passo min/km (formato `M:SS`)
- **`_cadence()`** — normalizza la cadenza: Garmin a volte restituisce passi per minuto *per un piede* (valore < 120), in tal caso moltiplica × 2

In [8]:
import json

RUNNING_TYPES = {"running", "trail_running", "treadmill_running"}

snapshot = {
    "generated_at": TODAY.isoformat(),
    "days": DAYS,
}

def save():
    OUTPUT_PATH.write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def _pace(speed_m_s):
    if not speed_m_s:
        return None
    s = int(1000 / speed_m_s)
    return f"{s // 60}:{s % 60:02d}"

def _cadence(raw):
    if raw is None:
        return None
    return int(raw * 2) if raw < 120 else int(raw)

save()
print("Snapshot inizializzato e salvato.")

Snapshot inizializzato e salvato.


## 4 — Attività di corsa (ultimi 30 giorni)

Scarica le attività degli ultimi `DAYS` giorni e filtra solo quelle di tipo **running** (corsa su strada, trail, tapis roulant). Per ogni attività vengono estratti: id, data, tipo, nome, distanza, durata, passo medio, FC media e massima, calorie, VO₂max, cadenza e lunghezza del passo.

Gli ID vengono tenuti in memoria perché servono nella cella successiva per scaricare i lap.

In [9]:
start_date = (TODAY - timedelta(days=DAYS)).isoformat()
end_date   = TODAY.isoformat()

raw_acts   = client.get_activities_by_date(start_date, end_date)
activities = []

for a in raw_acts:
    if a.get("activityType", {}).get("typeKey") not in RUNNING_TYPES:
        continue
    activities.append({
        "activity_id":               a.get("activityId"),
        "date":                      a.get("startTimeLocal", "")[:10],
        "type":                      a.get("activityType", {}).get("typeKey"),
        "name":                      a.get("activityName"),
        "distance_km":               round((a.get("distance") or 0) / 1000, 2),
        "duration_min":              round((a.get("duration") or 0) / 60, 1),
        "avg_pace_min_km":           _pace(a.get("averageSpeed")),
        "avg_hr":                    a.get("averageHR"),
        "max_hr":                    a.get("maxHR"),
        "calories":                  a.get("calories"),
        "vo2max":                    a.get("vO2MaxValue"),
        "aerobic_training_effect":   a.get("aerobicTrainingEffect"),
        "anaerobic_training_effect": a.get("anaerobicTrainingEffect"),
        "training_stress_score":     a.get("trainingStressScore"),
        "avg_cadence_spm":           _cadence(a.get("averageRunningCadenceInStepsPerMinute")),
        "max_cadence_spm":           _cadence(a.get("maxRunningCadenceInStepsPerMinute")),
        "avg_stride_length_m":       a.get("avgStrideLength"),
    })

snapshot["activities"] = activities
save()
print(f"Corse scaricate: {len(activities)}")

Corse scaricate: 12


## 5 — Splits (lap per ogni corsa)

Per ogni attività di tipo **running** (corsa su strada, trail, tapis roulant) scarica i singoli lap tramite l'endpoint dedicato `get_activity_splits()`. I lap sono nel campo `lapDTOs` della risposta.

Per ogni lap vengono estratti distanza, durata, passo medio, FC e cadenza (normalizzata) e dislivello. Il progresso viene stampato attività per attività per monitorare lo scaricamento.

In [11]:
running_ids = [
    a["activity_id"]
    for a in activities
    if a.get("type") in RUNNING_TYPES and a.get("activity_id")
]

splits = []
for act_id in running_ids:
    try:
        raw      = client.get_activity_splits(act_id)
        laps_raw = raw.get("lapDTOs") or raw.get("laps") or []
        laps_parsed = [
            {
                "lap_number":       lap.get("lapIndex"),
                "distance_km":      round((lap.get("distance") or 0) / 1000, 2),
                "duration_min":     round((lap.get("duration") or 0) / 60, 2),
                "pace_min_km":      _pace(lap.get("averageSpeed")),
                "avg_hr":           lap.get("averageHR"),
                "avg_cadence_spm":  _cadence(lap.get("averageRunCadence")),
                "elevation_gain_m": lap.get("elevationGain"),
            }
            for lap in laps_raw
        ]
        splits.append({"activity_id": act_id, "splits": laps_parsed})
        print(f"  OK  {act_id} — {len(laps_parsed)} lap")
    except Exception as e:
        splits.append({"activity_id": act_id, "error": str(e)})
        print(f"  ERR {act_id} — {e}")

snapshot["splits"] = splits
save()
print(f"\nSplits scaricati per {len(splits)} corse.")

  OK  22833301467 — 25 lap
  OK  22811195499 — 42 lap
  OK  22784338523 — 30 lap
  OK  22763588272 — 9 lap
  OK  22704702177 — 11 lap
  OK  22665003775 — 18 lap
  OK  22621145069 — 29 lap
  OK  22586020693 — 13 lap
  OK  22557784536 — 21 lap
  OK  22538860944 — 29 lap
  OK  22538101480 — 3 lap
  OK  22479064909 — 5 lap

Splits scaricati per 12 corse.


## 6 — Frequenza cardiaca a riposo

Il dato di **resting heart rate** viene letto giorno per giorno tramite `get_rhr_day()`. La risposta API è annidata in `allMetrics → metricsMap → WELLNESS_RESTING_HEART_RATE`; estraiamo il primo valore disponibile. I giorni senza dato vengono comunque inseriti con `null` per mantenere la serie completa; un counter separato distingue i fallimenti API dai semplici giorni senza misurazione.

In [13]:
resting_hr = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw = client.get_rhr_day(d)
        rhr = (
            raw.get("allMetrics", {})
               .get("metricsMap", {})
               .get("WELLNESS_RESTING_HEART_RATE", [{}])[0]
               .get("value")
        )
    except Exception:
        rhr = None
        errors += 1
    resting_hr.append({"date": d, "resting_hr": rhr})

snapshot["resting_hr"] = resting_hr
save()
ok = sum(1 for x in resting_hr if x['resting_hr'] is not None)
print(f"Resting HR scaricato: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

Resting HR scaricato: 30/30 giorni con dato


## 7 — Body Battery

La **Body Battery** rappresenta il livello di energia percepita dal dispositivo (scala 0–100). L'API restituisce sia totali giornalieri che il time-series `bodyBatteryValuesArray`. Per ogni giorno estraiamo:
- **`charged`** — totale punti caricati nella giornata (somma di tutte le ricariche)
- **`drained`** — totale punti scaricati nella giornata
- **`peak_level`** — livello massimo istantaneo raggiunto (0–100), tipicamente al risveglio
- **`min_level`** — livello minimo istantaneo raggiunto, di solito a fine giornata o durante stress elevato

In [ ]:
body_battery = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw = client.get_body_battery(d) or []
        if not raw:
            body_battery.append({"date": d, "charged": None, "drained": None,
                                 "peak_level": None, "min_level": None})
            continue

        item = raw[0] if isinstance(raw, list) else raw
        values = item.get("bodyBatteryValuesArray") or []
        levels = [v[1] for v in values if isinstance(v, (list, tuple)) and len(v) >= 2 and isinstance(v[1], (int, float))]

        body_battery.append({
            "date":       d,
            "charged":    item.get("charged"),
            "drained":    item.get("drained"),
            "peak_level": max(levels) if levels else None,
            "min_level":  min(levels) if levels else None,
        })
    except Exception:
        body_battery.append({"date": d, "charged": None, "drained": None,
                             "peak_level": None, "min_level": None})
        errors += 1

snapshot["body_battery"] = body_battery
save()
ok = sum(1 for x in body_battery if x['peak_level'] is not None)
print(f"Body Battery scaricata: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

## 8 — Sonno

I dati del sonno vengono letti tramite `get_sleep_data()`. Il payload principale è nel campo `dailySleepDTO` e include la durata totale e la suddivisione nelle fasi (profondo, leggero, REM, sveglio). Il **sleep score** si trova in `sleepScores → overall → value`.

In [17]:
def _h(seconds):
    return round(seconds / 3600, 2) if seconds is not None else None

def _m(seconds):
    return round(seconds / 60, 1) if seconds is not None else None

sleep = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw   = client.get_sleep_data(d)
        daily = raw.get("dailySleepDTO", {}) or {}
        sleep.append({
            "date":             d,
            "sleep_duration_h": _h(daily.get("sleepTimeSeconds")),
            "deep_sleep_min":   _m(daily.get("deepSleepSeconds")),
            "light_sleep_min":  _m(daily.get("lightSleepSeconds")),
            "rem_sleep_min":    _m(daily.get("remSleepSeconds")),
            "awake_min":        _m(daily.get("awakeSleepSeconds")),
            "sleep_score":      (daily.get("sleepScores") or {}).get("overall", {}).get("value"),
        })
    except Exception:
        sleep.append({"date": d, "sleep_duration_h": None, "deep_sleep_min": None,
                      "light_sleep_min": None, "rem_sleep_min": None,
                      "awake_min": None, "sleep_score": None})
        errors += 1

snapshot["sleep"] = sleep
save()
ok = sum(1 for x in sleep if x['sleep_duration_h'] is not None)
print(f"Sleep scaricato: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

Sleep scaricato: 28/30 giorni con dato


## 9 — HRV (Heart Rate Variability)

L'**HRV** misura la variabilità dell'intervallo tra battiti cardiaci e viene usato come indicatore del recupero e dello stato del sistema nervoso autonomo. Da `hrvSummary` estraiamo:
- **`hrv_weekly_avg`** — media mobile settimanale (più stabile, baseline)
- **`hrv_last_night`** — valore medio della notte precedente (più puntuale, mostra variazioni giornaliere)
- **`hrv_status`** — stato Garmin: `BALANCED`, `UNBALANCED`, `LOW`, ecc.

In [19]:
hrv = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw     = client.get_hrv_data(d) or {}
        summary = raw.get("hrvSummary") or {}
        hrv.append({
            "date":           d,
            "hrv_weekly_avg": summary.get("weeklyAvg"),
            "hrv_last_night": summary.get("lastNight"),
            "hrv_status":     summary.get("status"),
        })
    except Exception:
        hrv.append({"date": d, "hrv_weekly_avg": None, "hrv_last_night": None, "hrv_status": None})
        errors += 1

snapshot["hrv"] = hrv
save()
ok = sum(1 for x in hrv if x['hrv_weekly_avg'] is not None or x['hrv_last_night'] is not None)
print(f"HRV scaricato: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

HRV scaricato: 29/30 giorni con dato


## 10 — Stress

Il livello di **stress** viene calcolato dal dispositivo in base alla variabilità cardiaca durante la giornata. Per ogni giorno estraiamo:
- **`avg_stress`** / **`max_stress`** — livello medio e picco (scala 0–100)
- **`rest_stress_s`** / **`low_stress_s`** / **`med_stress_s`** / **`high_stress_s`** — durate in secondi per ciascuna fascia

> Nota: Garmin restituisce `-1` o `-2` quando non ha misurazioni (orologio non indossato). Questi valori vengono normalizzati a `None`.

In [20]:
def _nonneg(v):
    """Garmin restituisce -1/-2 per 'no data' → normalizza a None."""
    return v if isinstance(v, (int, float)) and v >= 0 else None

stress = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw = client.get_stress_data(d) or {}
        stress.append({
            "date":          d,
            "avg_stress":    _nonneg(raw.get("avgStressLevel")),
            "max_stress":    _nonneg(raw.get("maxStressLevel")),
            "rest_stress_s": raw.get("restStressDuration"),
            "low_stress_s":  raw.get("lowStressDuration"),
            "med_stress_s":  raw.get("mediumStressDuration"),
            "high_stress_s": raw.get("highStressDuration"),
        })
    except Exception:
        stress.append({"date": d, "avg_stress": None, "max_stress": None,
                       "rest_stress_s": None, "low_stress_s": None,
                       "med_stress_s": None, "high_stress_s": None})
        errors += 1

snapshot["stress"] = stress
save()
ok = sum(1 for x in stress if x['avg_stress'] is not None)
print(f"Stress scaricato: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

Stress scaricato: 30/30 giorni con dato


## 11 — Passi giornalieri

`get_steps_data()` restituisce una lista di intervalli intraday; i valori vengono sommati per ottenere il totale giornaliero.

In [21]:
steps = []
errors = 0
for i in range(DAYS):
    d = (TODAY - timedelta(days=i)).isoformat()
    try:
        raw = client.get_steps_data(d)
        if isinstance(raw, list) and raw:
            total = sum(x.get("steps", 0) for x in raw)
        else:
            total = None
        steps.append({"date": d, "steps": total})
    except Exception:
        steps.append({"date": d, "steps": None})
        errors += 1

snapshot["steps"] = steps
save()
ok = sum(1 for x in steps if x['steps'] is not None)
print(f"Steps scaricati: {ok}/{DAYS} giorni con dato" + (f" — {errors} errori API" if errors else ""))

Steps scaricati: 30/30 giorni con dato


## 12 — Training Plan

Scarica i workout pianificati nell'intervallo **−7 / +30 giorni** rispetto ad oggi. L'API `get_scheduled_workouts()` opera mese per mese, quindi vengono fetchati tutti i mesi che ricadono nell'intervallo e i risultati vengono filtrati per data.

> Nota: `get_training_plans()` restituisce una lista vuota anche con un piano Garmin Coach attivo — il metodo corretto è `get_scheduled_workouts(year, month)`.

In [22]:
plan_start = TODAY - timedelta(days=7)
plan_end   = TODAY + timedelta(days=30)

months_to_fetch = set()
cursor = plan_start.replace(day=1)
while cursor <= plan_end:
    months_to_fetch.add((cursor.year, cursor.month))
    cursor = (cursor.replace(day=28) + timedelta(days=4)).replace(day=1)

raw_items = []
errors = 0
for year, month in sorted(months_to_fetch):
    try:
        raw = client.get_scheduled_workouts(year, month)
        if isinstance(raw, list):
            raw_items.extend(raw)
        elif isinstance(raw, dict):
            raw_items.extend(raw.get("calendarItems") or raw.get("items") or [])
    except Exception:
        errors += 1

training_plan = []
for w in raw_items:
    if not isinstance(w, dict):
        continue
    w_date_str = w.get("date") or w.get("scheduledDate")
    if not w_date_str:
        continue
    try:
        w_date = date.fromisoformat(w_date_str[:10])
    except ValueError:
        continue
    if not (plan_start <= w_date <= plan_end):
        continue
    sport = w.get("sportType") or {}
    training_plan.append({
        "date":                   w_date_str[:10],
        "workout_id":             w.get("workoutId") or w.get("id"),
        "name":                   w.get("workoutName") or w.get("title"),
        "description":            w.get("description"),
        "type":                   sport.get("sportTypeKey") if isinstance(sport, dict) else w.get("itemType"),
        "estimated_duration_min": round((w.get("estimatedDurationInSecs") or 0) / 60, 1) or None,
        "estimated_distance_km":  round((w.get("estimatedDistanceInMeters") or 0) / 1000, 2) or None,
    })

training_plan.sort(key=lambda x: x["date"])
snapshot["training_plan"] = training_plan
save()
print(f"Training plan: {len(training_plan)} workout nell'intervallo {plan_start} → {plan_end}"
      + (f" — {errors} mesi con errore API" if errors else ""))

Training plan: 12 workout nell'intervallo 2026-05-03 → 2026-06-09


## 13 — Personal Records

I record personali sono uno snapshot del momento attuale (non storicizzati). `get_personal_record()` restituisce una lista in cui ogni elemento ha un `typeId` intero che identifica la categoria del PR. La mappa `PR_TYPE_ID` traduce gli ID nelle etichette leggibili (verificate contro l'app Garmin Connect IT).

I `value` hanno **unità di misura diverse** a seconda del tipo:
- record di tempo (1km, 1mile, 5km, 10km, mezza, maratona) → **secondi**
- `longest_run` → **metri**
- `daily/weekly/monthly_steps` → **conteggio passi**

Il campo `prStartTimeGmt` può essere stringa ISO o timestamp in millisecondi a seconda del tipo di PR; `_pr_date()` gestisce entrambi i casi.

In [34]:
from datetime import datetime, timezone

# Mappa typeId → label, verificata 2026-05-10 contro l'app Garmin Connect (utente IT).
PR_TYPE_ID = {
    1:  "1km",                  # tempo per 1 km (sec)
    2:  "1mile",                # tempo per 1 miglio (sec)
    3:  "5km",                  # tempo per 5 km (sec)
    4:  "10km",                 # tempo per 10 km (sec)
    5:  "half_marathon",        # tempo per mezza maratona (sec)
    6:  "marathon",             # tempo per maratona (sec)
    7:  "longest_run",          # distanza massima singola corsa (m)
    12: "daily_steps",
    13: "weekly_steps",
    14: "monthly_steps",
    15: "longest_goal_streak",  # giorni
    16: "current_goal_streak",  # giorni
}

def _pr_date(pr):
    """Preferisci prStartTimeLocal (no fuso) → fallback prStartTimeGmt."""
    for key in ("prStartTimeLocal", "prStartTimeGmt"):
        raw = pr.get(key)
        if raw is None:
            continue
        if isinstance(raw, str):
            return raw[:10]
        if isinstance(raw, (int, float)):
            return datetime.fromtimestamp(raw / 1000, tz=timezone.utc).date().isoformat()
    return None

personal_records = []
unknown_types = set()
try:
    raw_prs = client.get_personal_record()
    for pr in (raw_prs if isinstance(raw_prs, list) else []):
        type_id = pr.get("typeId")
        label   = PR_TYPE_ID.get(type_id)
        if label is None:
            label = f"type_{type_id}"
            unknown_types.add(type_id)
        personal_records.append({
            "type_id": type_id,
            "type":    label,
            "value":   pr.get("value"),
            "date":    _pr_date(pr),
        })
    pr_error = None
except Exception as e:
    pr_error = f"{type(e).__name__}: {e}"

snapshot["personal_records"] = personal_records
if pr_error:
    snapshot["personal_records_error"] = pr_error
else:
    snapshot.pop("personal_records_error", None)
save()

if pr_error:
    print(f"ERRORE personal records: {pr_error}")
else:
    print(f"Personal records scaricati: {len(personal_records)} record")
    for pr in personal_records:
        print(f"  {pr['type']:<22} {pr['value']}  ({pr['date']})")
    if unknown_types:
        print(f"\nTypeId sconosciuti — verifica su app Garmin e aggiungili a PR_TYPE_ID: {sorted(unknown_types)}")

Personal records scaricati: 11 record
  1km                    249.69000244140625  (2026-04-10)
  1mile                  443.5849914550781  (2026-04-17)
  5km                    1611.5279541015625  (2026-04-17)
  10km                   3359.962890625  (2026-04-29)
  half_marathon          8364.7392578125  (2026-05-10)
  longest_run            21210.439453125  (2026-05-10)
  daily_steps            24606.0  (2026-05-10)
  weekly_steps           98554.0  (2026-05-03)
  monthly_steps          304415.0  (2026-03-31)
  longest_goal_streak    5.0  (2026-03-06)
  current_goal_streak    1.0  (2026-05-10)


## 14 — Fitness Age

La **Fitness Age** è una stima dell'età biologica calcolata da Garmin sulla base di VO₂max, FC a riposo, BMI e minuti di attività intensa settimanale. Estraiamo:
- **`chronological_age`** — età anagrafica (per confronto)
- **`fitness_age`** — età fitness attuale
- **`previous_fitness_age`** — valore precedente (utile per vedere il trend)
- **`achievable_fitness_age`** — età fitness raggiungibile migliorando i componenti

E i quattro **componenti** che concorrono al calcolo (in `components.*.value`):
- **`rhr`** — frequenza cardiaca a riposo (bpm)
- **`bmi`** — Body Mass Index
- **`vigorous_days_avg`** — media giorni/settimana con attività intensa
- **`vigorous_minutes_avg`** — media minuti/settimana di attività intensa

Più **`last_updated`** — data dell'ultimo aggiornamento della stima.

In [35]:
fitness_age = {}
fa_error    = None
try:
    raw_fa     = client.get_fitnessage_data(TODAY.isoformat()) or {}
    components = raw_fa.get("components") or {}
    fitness_age = {
        "chronological_age":      raw_fa.get("chronologicalAge"),
        "fitness_age":            raw_fa.get("fitnessAge"),
        "previous_fitness_age":   raw_fa.get("previousFitnessAge"),
        "achievable_fitness_age": raw_fa.get("achievableFitnessAge"),
        "rhr":                    (components.get("rhr") or {}).get("value"),
        "bmi":                    (components.get("bmi") or {}).get("value"),
        "vigorous_days_avg":      (components.get("vigorousDaysAvg") or {}).get("value"),
        "vigorous_minutes_avg":   (components.get("vigorousMinutesAvg") or {}).get("value"),
        "last_updated":           raw_fa.get("lastUpdated", "")[:10] or None,
    }
    if all(v is None for v in fitness_age.values()):
        print(f"Attenzione: tutti i campi fitness_age sono null. Chiavi raw disponibili: {list(raw_fa.keys())}")
except Exception as e:
    fa_error = f"{type(e).__name__}: {e}"

snapshot["fitness_age"] = fitness_age
if fa_error:
    snapshot["fitness_age_error"] = fa_error
else:
    snapshot.pop("fitness_age_error", None)
save()

if fa_error:
    print(f"ERRORE fitness age: {fa_error}")
else:
    print("Fitness Age:")
    for k, v in fitness_age.items():
        print(f"  {k:<24} {v}")

Fitness Age:
  chronological_age        27
  fitness_age              20.14335036802265
  previous_fitness_age     20.14335036802265
  achievable_fitness_age   18.0
  rhr                      52
  bmi                      23.2
  vigorous_days_avg        2.5
  vigorous_minutes_avg     149.6
  last_updated             2026-05-10


## Riepilogo finale

Tutte le sezioni sono state scaricate. La cella seguente stampa un riepilogo del contenuto dello snapshot e conferma il percorso del file salvato.

In [36]:
def _count_with_data(records, *value_keys):
    """Conta record con almeno una chiave non-None tra value_keys."""
    if not isinstance(records, list):
        return 0
    return sum(
        1 for r in records
        if isinstance(r, dict) and any(r.get(k) is not None for k in value_keys)
    )

print("=" * 60)
print(f" Snapshot salvato in: {OUTPUT_PATH}")
print(f" Dimensione file:     {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")
print(f" Generato il:         {snapshot['generated_at']}")
print(f" Finestra storica:    {DAYS} giorni")
print("=" * 60)

# Sezioni con metriche giornaliere — mostra "con dato / totale"
daily_sections = [
    ("resting_hr",   ["resting_hr"]),
    ("body_battery", ["peak_level"]),
    ("sleep",        ["sleep_duration_h"]),
    ("hrv",          ["hrv_weekly_avg", "hrv_last_night"]),
    ("stress",       ["avg_stress"]),
    ("steps",        ["steps"]),
]
print("\nSerie giornaliere:")
for key, value_keys in daily_sections:
    records = snapshot.get(key, [])
    with_data = _count_with_data(records, *value_keys)
    print(f"  {key:<22} {with_data:>3}/{len(records):<3} giorni con dato")

# Sezioni a conteggio semplice
print("\nAltre sezioni:")
for key in ("activities", "splits", "training_plan", "personal_records"):
    val = snapshot.get(key, [])
    print(f"  {key:<22} {len(val):>3} record")

# Errori sezioni (se presenti)
error_keys = [k for k in snapshot if k.endswith("_error")]
if error_keys:
    print("\n⚠ Errori:")
    for k in error_keys:
        print(f"  {k:<22} {snapshot[k]}")

# Fitness age — riepilogo completo
print("\nFitness Age:")
fa = snapshot.get("fitness_age", {}) or {}
for k, v in fa.items():
    print(f"  {k:<24} {v}")

 Snapshot salvato in: c:\Users\feder\Script\data\garmin_2026-05-10.json
 Dimensione file:     94.1 KB
 Generato il:         2026-05-10
 Finestra storica:    30 giorni

Serie giornaliere:
  resting_hr              30/30  giorni con dato
  body_battery            30/30  giorni con dato
  sleep                   28/30  giorni con dato
  hrv                     29/30  giorni con dato
  stress                  30/30  giorni con dato
  steps                   30/30  giorni con dato

Altre sezioni:
  activities              12 record
  splits                  12 record
  training_plan           12 record
  personal_records        11 record

Fitness Age:
  chronological_age        27
  fitness_age              20.14335036802265
  previous_fitness_age     20.14335036802265
  achievable_fitness_age   18.0
  rhr                      52
  bmi                      23.2
  vigorous_days_avg        2.5
  vigorous_minutes_avg     149.6
  last_updated             2026-05-10
